# Notebook 03: The Transformer Decoder

## Why This Matters

The transformer decoder is the engine behind GPT-4, Claude, LLaMA, Mistral, Gemma, and every modern large language model. Understanding the decoder means understanding why autoregressive generation works, what limits the speed of inference, and how techniques like KV caching reduce cost by orders of magnitude. Every company building LLM-powered products is constrained by inference cost — and KV caching is the most important optimization that makes real-time generation economically viable. This notebook builds a decoder from scratch, implements KV caching, and demonstrates the full greedy decoding loop.

## Background

The decoder is more complex than the encoder because it has two different modes:
1. **Training (teacher forcing):** All target tokens are available simultaneously; the model predicts the next token at every position in parallel. A causal mask prevents position $i$ from attending to position $j > i$.
2. **Inference (autoregressive):** Tokens are generated one at a time; each new token is fed back as input for the next step.

**Two architecture variants:**
- **Encoder-decoder** (T5, BART, original transformer): decoder has self-attention + cross-attention + FFN
- **Decoder-only** (GPT, LLaMA): no cross-attention; pure causal self-attention + FFN. This has become dominant for general-purpose LLMs.

**Why decoder-only dominates:** Decoder-only models can be trained on massive text corpora without parallel source-target pairs. Combined with instruction tuning and RLHF, they generalize to virtually any task. The encoder-decoder architecture requires task-specific paired data and doesn't benefit as much from scale.

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
torch.manual_seed(42)
print("Ready.")

## 1. Causal Self-Attention: The Autoregressive Constraint

The key architectural constraint for a language model: **position $i$ can only attend to positions $\leq i$**. This is implemented with the upper-triangular causal mask:

$$\text{mask}_{ij} = \begin{cases} 0 & \text{if } j \leq i \\ -\infty & \text{if } j > i \end{cases}$$

**Why does this matter for training?** During training, we feed the entire target sequence at once and compute the loss at all positions simultaneously. Without the causal mask, the model at position $i$ would see token $i+1$ when predicting token $i$ — the task becomes trivial copying, and the model learns nothing.

**Interview question:** "How does the causal mask enable parallel training but sequential inference?"
**Answer:** During training, the mask is applied to all positions simultaneously via matrix operations — the forward pass for the entire sequence runs in one shot. At inference, we generate token by token (each step conditions on all previous tokens), which is inherently sequential. KV caching bridges this gap by storing the keys and values from previous steps.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))
    attn_weights = F.softmax(scores, dim=-1)
    return torch.matmul(attn_weights, V), attn_weights

def make_causal_mask(seq_len, device='cpu'):
    return torch.triu(torch.ones(seq_len, seq_len, device=device),
                      diagonal=1).bool().unsqueeze(0).unsqueeze(0)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def split_heads(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.n_heads, self.d_k).transpose(1, 2)

    def forward(self, query, key, value, mask=None):
        B, T_q, _ = query.shape
        Q = self.split_heads(self.W_q(query))
        K = self.split_heads(self.W_k(key))
        V = self.split_heads(self.W_v(value))
        attn_out, attn_weights = scaled_dot_product_attention(Q, K, V, mask=mask)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T_q, self.d_model)
        return self.W_o(attn_out), attn_weights

print("Core attention components loaded.")

## 2. KV Cache: Transforming O(n²) to O(n) per Step

Without KV caching, each new token generation step recomputes keys and values for ALL previous tokens from scratch. For a sequence of length $n$, step $n$ costs $O(n \cdot d)$ compute just to recompute prior K,V pairs — and total generation cost is $O(n^2)$.

**The insight:** Keys and values for position $i$ depend only on the token at position $i$ (after the linear projection). They don't change when new tokens are added. So we can cache them:

1. First forward pass ("prefill"): compute K, V for all prompt tokens simultaneously
2. Each subsequent step: compute only the new token's Q, K, V; append new K, V to cache; run attention over all cached K, V

**Memory cost:** KV cache size = $2 \times n_{\text{layers}} \times n_{\text{heads}} \times T \times d_k \times \text{dtype\_bytes}$

For LLaMA-7B with 32 layers, 32 heads, $d_k=128$, FP16, 4096 tokens:
$2 \times 32 \times 32 \times 4096 \times 128 \times 2 \approx 2 \text{ GB}$

This is why KV cache memory is a hard constraint on batch size and context length in production systems.

In [ ]:
class CausalSelfAttentionWithKVCache(nn.Module):
    """Causal self-attention with optional KV caching for inference."""
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        # KV cache storage
        self._cache_k = None
        self._cache_v = None

    def split_heads(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.n_heads, self.d_k).transpose(1, 2)

    def reset_cache(self):
        self._cache_k = None
        self._cache_v = None

    def forward(self, x, use_cache=False):
        B, T_new, _ = x.shape

        # Project new tokens
        Q = self.split_heads(self.W_q(x))           # (B, h, T_new, d_k)
        new_K = self.split_heads(self.W_k(x))        # (B, h, T_new, d_k)
        new_V = self.split_heads(self.W_v(x))        # (B, h, T_new, d_k)

        if use_cache and self._cache_k is not None:
            # Append new K, V to cached ones
            K = torch.cat([self._cache_k, new_K], dim=2)  # (B, h, T_total, d_k)
            V = torch.cat([self._cache_v, new_V], dim=2)
        else:
            K, V = new_K, new_V

        # Update cache
        if use_cache:
            self._cache_k = K.detach()
            self._cache_v = V.detach()

        T_total = K.size(2)

        # Causal mask: Q positions can only attend to past K positions
        # Q shape: (B, h, T_new, d_k), K shape: (B, h, T_total, d_k)
        # For generation (T_new=1), no mask needed (only one query attending to all past)
        if T_new > 1:
            # Training / prefill: full causal mask
            mask = torch.triu(
                torch.ones(T_new, T_total, device=x.device), diagonal=T_total - T_new + 1
            ).bool().unsqueeze(0).unsqueeze(0)
        else:
            mask = None  # Single query token can attend to all cached keys

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))
        attn_weights = F.softmax(scores, dim=-1)
        out = torch.matmul(attn_weights, V)
        out = out.transpose(1, 2).contiguous().view(B, T_new, self.d_model)
        return self.W_o(out)

# Demonstrate KV cache
B, d_model, n_heads = 1, 128, 8
attn = CausalSelfAttentionWithKVCache(d_model, n_heads)

# Simulate generation: process 5 tokens one at a time with caching
print("Generating with KV cache:")
for step in range(5):
    x_step = torch.randn(B, 1, d_model)
    out = attn(x_step, use_cache=True)
    cache_size = attn._cache_k.shape[2]
    print(f"  Step {step+1}: output shape {out.shape}, cache size {cache_size}")

attn.reset_cache()
print("Cache reset.")

## 3. GPT-Style Decoder Layer

A decoder-only (GPT-style) layer has no cross-attention — just causal self-attention + FFN with Pre-LN and residual connections. This is the building block of every modern LLM.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=None, dropout=0.1):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.gelu(self.linear1(x))))

class GPTDecoderLayer(nn.Module):
    """Decoder-only layer (GPT-style): causal self-attn + FFN, Pre-LN."""
    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        self.self_attn = CausalSelfAttentionWithKVCache(d_model, n_heads)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def reset_cache(self):
        self.self_attn.reset_cache()

    def forward(self, x, use_cache=False):
        residual = x
        x_normed = self.norm1(x)
        attn_out = self.self_attn(x_normed, use_cache=use_cache)
        x = residual + self.dropout(attn_out)

        residual = x
        x = residual + self.dropout(self.ffn(self.norm2(x)))
        return x

class MiniGPT(nn.Module):
    """Minimal GPT-style decoder-only transformer."""
    def __init__(self, vocab_size, d_model, n_heads, n_layers,
                 max_len=512, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.layers = nn.ModuleList([
            GPTDecoderLayer(d_model, n_heads, dropout=dropout)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        # Weight tying: token embedding and LM head share weights
        self.lm_head.weight = self.token_emb.weight

    def reset_cache(self):
        for layer in self.layers:
            layer.reset_cache()

    def forward(self, input_ids, use_cache=False, start_pos=0):
        B, T = input_ids.shape
        positions = torch.arange(start_pos, start_pos + T, device=input_ids.device)
        x = self.token_emb(input_ids) + self.pos_emb(positions)
        for layer in self.layers:
            x = layer(x, use_cache=use_cache)
        x = self.norm(x)
        return self.lm_head(x)  # (B, T, vocab_size)

vocab_size = 500
model = MiniGPT(vocab_size, d_model=128, n_heads=8, n_layers=4).to(device)
tokens = torch.randint(0, vocab_size, (2, 20)).to(device)
logits = model(tokens)
print(f"GPT forward pass: {logits.shape}")  # (2, 20, 500)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

## 4. Greedy Decoding Loop

At inference, we generate tokens autoregressively. The simplest strategy is **greedy decoding**: at each step, pick the token with the highest logit (argmax). More sophisticated strategies include sampling (temperature, top-k, top-p) and beam search.

In [ ]:
@torch.no_grad()
def greedy_decode(model, prompt_ids, max_new_tokens=20, device='cpu'):
    """
    Greedy autoregressive decoding with KV cache.
    prompt_ids: (1, T_prompt) initial token sequence
    """
    model.eval()
    model.reset_cache()
    input_ids = prompt_ids.to(device)

    # Prefill: process entire prompt at once (builds KV cache)
    B, T_prompt = input_ids.shape
    _ = model(input_ids, use_cache=True, start_pos=0)

    generated = []
    # Generate new tokens one at a time
    last_token = input_ids[:, -1:]  # (B, 1)

    for step in range(max_new_tokens):
        current_pos = T_prompt + step
        logits = model(last_token, use_cache=True, start_pos=current_pos)
        # logits: (B, 1, vocab_size) — greedy: argmax over vocab
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)  # (B, 1)
        generated.append(next_token.item())
        last_token = next_token

    model.reset_cache()
    return generated

model_cpu = MiniGPT(vocab_size, d_model=128, n_heads=8, n_layers=4)
prompt = torch.randint(0, vocab_size, (1, 5))
print(f"Prompt tokens: {prompt[0].tolist()}")
generated = greedy_decode(model_cpu, prompt, max_new_tokens=10)
print(f"Generated tokens: {generated}")
print("(Random model = random tokens; real models would produce coherent text)")

## 5. Why Decoder-Only Dominates

**The three forces that made decoder-only the winning architecture:**

1. **Scale + data efficiency:** Decoder-only models can be pre-trained on raw text (any corpus), while encoder-decoder requires paired (source, target) data. The internet has vastly more raw text than parallel corpora.

2. **Instruction following + RLHF:** The InstructGPT/ChatGPT breakthrough was instruction fine-tuning + RLHF on top of GPT-3. This works naturally with decoder-only (the model generates completions following instructions). The RLHF pipeline is simpler for a single model than for encoder-decoder.

3. **Emergent reasoning:** Chain-of-thought reasoning, in-context learning, and emergent capabilities appear at scale in decoder-only models. The ability to "think out loud" by generating intermediate reasoning steps emerges naturally from the autoregressive objective.

**When encoder-decoder still wins:**
- Long-input → short-output (summarization, translation) where you want full bidirectional encoding of a long document
- Tasks with very long source but structured output (T5 for structured prediction)
- Lower inference cost for fixed-output tasks (encoder runs once, decoder is short)

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| Causal mask | Upper-triangular -inf; prevents seeing future tokens; enables parallel training |
| Teacher forcing | Training feeds ground-truth targets; inference feeds model's own predictions |
| KV cache | Store K,V for previous positions; reduces generation from $O(n^2)$ to $O(n)$ per step |
| KV cache memory | $2 \times L \times H \times T \times d_k \times \text{bytes}$; often 1-4 GB for 7B models |
| Decoder-only (GPT) | No cross-attention; causal self-attn + FFN; trains on raw text |
| Encoder-decoder | Cross-attention connects encoder to decoder; best for seq2seq with long inputs |
| Weight tying | LM head shares weights with token embedding; reduces params, improves perplexity |
| Greedy decoding | Argmax at each step; fast but repetitive; sampling adds diversity |
| Autoregressive generation | Each token conditions on all previous; sequential at inference |